In [12]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [13]:
article = """
[단독] 스마트시티 프로젝트 '드림타운', 3년째 표류... 예산 낭비 논란
입력 2025.10.28 14:32 | 수정 2025.10.28 15:18
김태영 기자 (taeyoung.kim@newsexample.com)
정부가 야심차게 추진했던 스마트시티 프로젝트 '드림타운'이 착공 3년이 지난 현재까지도 완공 목표의 30%도 채우지 못한 채 표류하고 있어 세금 낭비 논란이 일고 있다.
국토교통부에 따르면 2022년 9월 착공한 경기도 화성시 일대 드림타운 프로젝트는 당초 2026년 말 완공을 목표로 총 사업비 2조 3천억 원을 투입하기로 했다. 하지만 현재까지 집행된 예산은 9,500억 원에 달하는 반면, 실제 공정률은 28%에 불과한 것으로 확인됐다.
특히 문제가 되는 것은 핵심 시설인 AI 통합관제센터와 자율주행 도로 인프라가 전혀 구축되지 않았다는 점이다. 사업 초기 화려하게 발표됐던 '세계 최초 완전 자율주행 도시', 'AI 기반 에너지 자급자족 도시'라는 슬로건은 현재 무색해진 상태다.
한 정부 관계자는 "기술 개발 지연과 민간 투자자 확보 실패가 주요 원인"이라고 해명했지만, 야당과 시민단체는 "처음부터 실현 불가능한 계획을 홍보용으로만 남발했다"며 강하게 비판하고 있다.
실제로 본지 취재 결과, 드림타운 프로젝트에 참여하기로 했던 대기업 3곳 중 2곳이 이미 사업에서 손을 뗀 것으로 드러났다. 한 대기업 관계자는 "정부의 계획이 너무 비현실적이고 수익성도 불투명해 투자를 회수할 수밖에 없었다"고 귀띔했다.
더욱 심각한 것은 이미 투입된 9,500억 원 중 상당 부분이 용역비와 컨설팅 비용으로 지출됐다는 점이다. 감사원 자료에 따르면 전체 집행 예산의 42%인 약 4,000억 원이 각종 용역과 연구비로 사용됐으며, 실제 건설에 투입된 금액은 절반에도 미치지 못했다.
환경운동연합 이수진 사무총장은 "드림타운 부지 조성 과정에서 생태보호구역이 훼손됐고, 주민 동의 절차도 제대로 거치지 않았다"며 "결국 누구를 위한 사업인지 의문"이라고 지적했다.
국회 국토교통위원회 소속 박민수 의원(더불어민주당)은 "전형적인 전시행정의 실패 사례"라며 "사업 전면 재검토와 책임자 문책이 필요하다"고 목소리를 높였다.
국토부는 다음 달 중 민관합동 TF를 구성해 사업 재조정안을 마련하겠다는 입장이지만, 이미 쏟아부은 예산을 고려할 때 중단도, 지속도 어려운 진퇴양난에 빠진 상황이다.
김태영 기자 taeyoung.kim@newsexample.com
"""

In [14]:
#실습용 AOAI 환경변수 읽기
import os

AOAI_ENDPOINT=os.getenv("AOAI_ENDPOINT")
AOAI_API_KEY=os.getenv("AOAI_API_KEY")
AOAI_DEPLOY_GPT4O=os.getenv("AOAI_DEPLOY_GPT4O")
AOAI_DEPLOY_GPT4O_MINI=os.getenv("AOAI_DEPLOY_GPT4O_MINI")
AOAI_DEPLOY_EMBED_3_LARGE=os.getenv("AOAI_DEPLOY_EMBED_3_LARGE")
AOAI_DEPLOY_EMBED_3_SMALL=os.getenv("AOAI_DEPLOY_EMBED_3_SMALL")
AOAI_DEPLOY_EMBED_ADA=os.getenv("AOAI_DEPLOY_EMBED_ADA")

In [15]:
from openai import AzureOpenAI
import json
client = AzureOpenAI(
  azure_endpoint = AOAI_ENDPOINT,
  api_key=AOAI_API_KEY,
  api_version="2024-10-21"
)

OpenAIError: Missing credentials. Please pass one of `api_key`, `azure_ad_token`, `azure_ad_token_provider`, or the `AZURE_OPENAI_API_KEY` or `AZURE_OPENAI_AD_TOKEN` environment variables.

In [ ]:
# 1. 의도 분석 스키마 (enum 사용)
intent_schema = {
    "name": "intent_analysis_schema",
    "schema": {
        "type": "object",
        "properties": {
            "intent": {
                "type": "string",
                "enum": ["title_extraction", "article_summary", "sentiment_analysis", "general_conversation"]
            }
        },
        "required": ["intent"],
        "additionalProperties": False
    }
}

def analyze_intent(user_input: str) -> dict:
    """
    1. 사용자 의도 분석 함수
    사용자의 쿼리가 어떤 작업을 원하는지 분석합니다.
    """
    INTENT_SYSTEM_PROMPT = """당신은 사용자의 의도를 분석하는 AI입니다.
    사용자의 입력을 분석하여 다음 중 정확히 하나의 의도를 선택하세요:
    - title_extraction: 텍스트에서 제목을 찾아달라는 요청
    - article_summary: 긴 텍스트를 요약해달라는 요청
    - sentiment_analysis: 텍스트의 긍정/부정을 평가해달라는 요청
    - general_conversation: 위 작업과 관련 없는 일반적인 대화
    """

    response = client.chat.completions.create(
        model=AOAI_DEPLOY_GPT4O,
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": intent_schema,
        },
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
query = f"{article} 이 기사의 제목을 추출해줘."
print(analyze_intent(query))

In [ ]:
# 2. 기사 제목 추출 스키마
title_extraction_schema = {
    "name": "title_extraction_schema",
    "schema": {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "has_title": {"type": "boolean"}
        },
        "required": ["title", "has_title"],
        "additionalProperties": False
    }
}

def extract_title(user_input: str) -> dict:
    """
    2. 기사 제목 추출 함수
    텍스트에서 제목과 부제목을 추출합니다.
    """

    TITLE_SYSTEM_PROMPT = """당신은 텍스트에서 제목을 추출하는 전문가입니다.
    주어진 텍스트에서 주요 제목과 부제목을 찾아내세요.
    제목이 명확하지 않으면 텍스트의 핵심 주제를 제목으로 생성하세요.
    """

    response = client.chat.completions.create(
        model=AOAI_DEPLOY_GPT4O,
        messages=[
            {"role": "system", "content": TITLE_SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": title_extraction_schema,
        },
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
# 3. 기사 본문 요약 스키마
summary_schema = {
    "name": "summary_schema",
    "schema": {
        "type": "object",
        "properties": {
            "summary": {"type": "string"},
            "key_points": {
                "type": "array",
                "items": {"type": "string"}
            },
            "word_count": {"type": "integer"}
        },
        "required": ["summary", "key_points", "word_count"],
        "additionalProperties": False
    }
}

def summarize_article(user_input: str) -> dict:
    """
    3. 기사 본문 요약 함수
    긴 텍스트를 핵심 포인트 중심으로 요약합니다.
    """

    SUMMARY_SYSTEM_PROMPT = """당신은 텍스트 요약 전문가입니다.
    주어진 텍스트를 간결하게 요약하고, 핵심 포인트를 3-5개로 추출하세요.
    요약문은 2-3문장으로 작성하세요.
    """
    response = client.chat.completions.create(
        model=AOAI_DEPLOY_GPT4O,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": summary_schema,
        },
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
# 4. 기사 긍부정 평가 스키마
sentiment_schema = {
    "name": "sentiment_schema",
    "schema": {
        "type": "object",
        "properties": {
            "sentiment_label": {
                "type": "string",
                "enum": ["positive", "negative", "neutral"]
            },
            "reason": {"type": "string"}
        },
        "required": ["sentiment_label", "reason"],
        "additionalProperties": False
    }
}
def analyze_sentiment(user_input: str) -> dict:
    """
    4. 기사 긍부정 평가 함수
    텍스트의 감정을 분석하고 긍정/부정/중립을 판단합니다.
    """
    SENTIMENT_SYSTEM_PROMPT = """당신은 텍스트의 감정을 분석하는 전문가입니다.
    주어진 텍스트의 전반적인 톤을 분석하여:
    - positive: 긍정적 감정이 우세한 경우
    - negative: 부정적 감정이 우세한 경우
    - neutral: 중립적이거나 사실 전달만 하는 경우
    """
    response = client.chat.completions.create(
        model=AOAI_DEPLOY_GPT4O,
        messages=[
            {"role": "system", "content": SENTIMENT_SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": sentiment_schema,
        },
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
# 5. 일반 대화 스키마
general_conversation_schema = {
    "name": "general_conversation_schema",
    "schema": {
        "type": "object",
        "properties": {
            "response": {"type": "string"},
            "is_question": {"type": "boolean"},
            "tone": {
                "type": "string",
                "enum": ["friendly", "professional", "casual", "formal"]
            }
        },
        "required": ["response", "is_question", "tone"],
        "additionalProperties": False
    }
}
def general_chat(user_input: str) -> dict:
    """
    5. 일반 대화 함수
    일반적인 대화에 응답합니다.
    """

    GENERAL_SYSTEM_PROMPT = """당신은 친근하고 도움이 되는 AI 어시스턴트입니다.
    사용자의 질문이나 대화에 자연스럽게 응답하세요.
    tone은 대화의 분위기에 맞게 선택하세요.
    """
    response = client.chat.completions.create(
        model=AOAI_DEPLOY_GPT4O,
        messages=[
            {"role": "system", "content": GENERAL_SYSTEM_PROMPT},
            {"role": "user", "content": user_input}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": general_conversation_schema,
        },
    )

    return json.loads(response.choices[0].message.content)

In [ ]:
# ============================================
# 메인 워크플로우 함수
# ============================================

def process_user_query(user_input: str):
    """
    사용자 쿼리를 받아 적절한 함수로 라우팅하여 처리하는 메인 워크플로우
    """
    print(f"\n{'='*60}")
    print(f"사용자 입력: {user_input}")
    print(f"{'='*60}\n")

    # Step 1: 의도 분석
    print("🔍 Step 1: 의도 분석 중...")
    intent_result = analyze_intent(user_input)
    intent = intent_result["intent"]
    print(f"분석 결과: {intent}\n")

    # Step 2: 의도에 따라 적절한 함수 호출
    if intent == "title_extraction":
        print("Step 2: 제목 추출 실행...")
        result = extract_title(user_input)
        print(f"결과: {json.dumps(result, ensure_ascii=False, indent=2)}")

    elif intent == "article_summary":
        print("Step 2: 본문 요약 실행...")
        result = summarize_article(user_input)
        print(f"결과: {json.dumps(result, ensure_ascii=False, indent=2)}")

    elif intent == "sentiment_analysis":
        print("Step 2: 감정 분석 실행...")
        result = analyze_sentiment(user_input)
        print(f"결과: {json.dumps(result, ensure_ascii=False, indent=2)}")

    elif intent == "general_conversation":
        print("Step 2: 일반 대화 처리 중...")
        result = general_chat(user_input)
        print(f"결과: {json.dumps(result, ensure_ascii=False, indent=2)}")

    else:
        print("의도를 파악할 수 없습니다.")
        result = None

In [ ]:
user = f"""
이 기사가 긍정적으로 쓰였나?
{article}
"""
process_user_query(user)